In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

 # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/sample_submission.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/train.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test.csv
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_49.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_90.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_77.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_66.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_54.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_42.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_81.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/test/audio_72.wav
/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final

In [5]:
import torch
import librosa
import numpy as np
import pandas as pd
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import parselmouth
from parselmouth.praat import call

In [6]:
file_dir = "/kaggle/input/competitions/shl-hiring-assessment-2026/Dataset_Final/"


In [7]:
processor = WhisperProcessor.from_pretrained("openai/whisper-medium")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")   

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

In [8]:
def transcribe(file_path):
    audio, _ = librosa.load(file_path, sr=16000)

    input_features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features

    with torch.no_grad():
        output_ids = model.generate(input_features)

    return processor.batch_decode(output_ids, skip_special_tokens=True)[0]

In [ ]:
def extract_features(file_path, transcription=None):
    audio, sr = librosa.load(file_path, sr=16000)
    sound = parselmouth.Sound(file_path)

    features = {}

    # --- MFCC ---
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    features["mfcc_mean"] = mfcc.mean(axis=1).tolist()
    features["mfcc_std"]  = mfcc.std(axis=1).tolist()

    # --- Pitch (F0) ---
    pitch = sound.to_pitch(time_step=0.01, pitch_floor=50, pitch_ceiling=500)
    f0 = pitch.selected_array["frequency"]
    f0_voiced = f0[f0 > 0]
    features["pitch_mean"]   = float(np.mean(f0_voiced)) if len(f0_voiced) else 0
    features["pitch_std"]    = float(np.std(f0_voiced)) if len(f0_voiced) else 0
    features["pitch_min"]    = float(np.min(f0_voiced)) if len(f0_voiced) else 0
    features["pitch_max"]    = float(np.max(f0_voiced)) if len(f0_voiced) else 0

    # --- Energy (RMS) ---
    rms = librosa.feature.rms(y=audio)[0]
    features["energy_mean"] = float(np.mean(rms))
    features["energy_std"]  = float(np.std(rms))

    # --- VAD: speech segments, pauses, silence ---
    speech_segments = librosa.effects.split(audio, top_db=30)  # (start, end) in samples
    speech_durations = [(e - s) / sr for s, e in speech_segments]
    features["speech_duration"] = float(sum(speech_durations))
    features["num_voiced_segments"] = len(speech_segments)

    # Pauses (gaps between speech segments)
    pauses = []
    for i in range(len(speech_segments) - 1):
        gap = (speech_segments[i+1][0] - speech_segments[i][1]) / sr
        if gap > 0.1:  # >100ms counts as a pause
            pauses.append(gap)
    features["num_pauses"] = len(pauses)
    features["pause_mean"] = float(np.mean(pauses)) if pauses else 0
    features["pause_max"]  = float(np.max(pauses)) if pauses else 0

    # Total silence
    total_duration = len(audio) / sr
    features["silence_duration"] = total_duration - features["speech_duration"]

    # --- Speaking rate (words/sec, needs transcription) ---
    if transcription:
        words = len(transcription.split())
        features["speaking_rate"] = words / features["speech_duration"] if features["speech_duration"] > 0 else 0
    else:
        features["speaking_rate"] = None  # needs Whisper transcription

    # --- Articulation features (jitter, shimmer, HNR) ---
    point_process = call(sound, "To PointProcess (periodic, cc)", 50, 500)
    features["jitter"] = call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
    features["shimmer"] = call([sound, point_process], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
    harmonicity = sound.to_harmonicity(time_step=0.01)
    features["hnr"] = harmonicity.get_mean()

    # --- Formants (F1-F3, articulation-related) ---
    formants = sound.to_formant_burg(time_step=0.01)
    for i in range(1, 4):
        vals = formants.selected_array[f"frequency{i}"]
        vals = vals[vals > 0]
        features[f"formant_f{i}_mean"] = float(np.mean(vals)) if len(vals) else 0

    return features

In [9]:
def process_audio(file_path):
    # 1. Transcribe once
    transcription = transcribe(file_path)

    # 2. Extract acoustic + speech features
    features = extract_features(
        file_path,
        transcription=transcription
    )

    # 3. Add transcription separately
    features["transcription"] = transcription
    features["file_path"] = file_path

    return features